# Distribution System State Estimation Using Wavelet Decomposition with NFPP Sodium-Ion BESS Performance Evaluation

This notebook implements the complete research pipeline for the DFN-based BESS optimization, dynamic transient simulation under partial observability, and rigorous wavelet-domain statistical state estimation.

In [ ]:
# Automate Wine installation if missing (required for Windows ATP-EMTP binaries on Linux research runtime)
import subprocess
try:
    subprocess.run(["wine", "--version"], check=True, capture_output=True)
    print("Wine is already installed on the research runtime.")
except Exception:
    print("Wine is missing. Installing Wine and i386 multiarch support...")
    subprocess.run("sudo dpkg --add-architecture i386 && sudo apt-get update && sudo apt-get install -y wine wine32:i386", shell=True)
    print("Wine successfully installed.")

import os
import sys

# Environment Setup and Namespace Package Support
root_dir = os.path.abspath(os.getcwd())
while root_dir and not any(os.path.exists(os.path.join(root_dir, d)) for d in ['nfpp_sodium_ion', 'docs']):
    parent = os.path.dirname(root_dir)
    if parent == root_dir:
        break
    root_dir = parent

if root_dir and os.path.exists(root_dir):
    nfpp_dir = os.path.join(root_dir, 'nfpp_sodium_ion')
    src_dir = os.path.join(root_dir, 'src')
    for p in [root_dir, nfpp_dir]:
        if p not in sys.path:
            sys.path.insert(0, p)
    import src
    if hasattr(src, '__path__') and src_dir not in src.__path__:
        src.__path__.append(src_dir)

!pip install pybamm numpy scipy pywavelets pandas opendssdirect OpenDSSDirect.py pyatp atp-utils matplotlib requests mp-api pymatgen pymoo mpi4py pint ufl
!add-apt-repository -y ppa:fenics-packages/fenics
!apt update
!apt install -y fenicsx
import pybamm
import numpy as np
import scipy
import pywt
import pandas as pd
import matplotlib.pyplot as plt
print("Environment initialized successfully.")

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()

print("\n--- OPTIMIZATION RESULTS ---")
print("Optimized Design Variables per Objective:")
for obj, specs in optimized_res.get("opt_designs_per_objective", {}).items():
    print(f"\nObjective: {obj.capitalize()}")
    for k, v in specs.items():
        print(f"  {k:40s}: {v:12.6e}")

print("\nSelected Integrated Design Variables:")
for k, v in optimized_res.get("design_specs_representative", {}).items():
    print(f"  {k:40s}: {v:12.6e}")

print("\n--- OPTIMAL CANDIDATE: QM DATA & DERIVED CELL PARAMETERS ---")
mats = optimized_res.get("materials", {})
deltas = optimized_res.get("combined_deltas_representative", {})
for cat in ["cathode", "electrolyte"]:
    print(f"\n{cat.capitalize()} Material:")
    m_data = mats.get(cat, {})
    print(f"  Name: {m_data.get('name') or m_data.get('salt')}")
    print(f"  Formula: {m_data.get('formula')}")
    print("  QM/Physics Properties:")
    for pk, pv in m_data.get("properties", {}).items():
        print(f"    {pk:25s}: {pv}")

print("\nMapping to PyBaMM Parameter Deltas:")
for category, props in deltas.items():
    print(f"  [{category.upper()}]")
    for pk, pv in props.items():
        print(f"    {pk:45s}: {pv:+.4e}")

## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

print("\nStage 3.1: Running BESS Robustness Evaluation...")
from src.simulation.tests import BESSEvaluator
bess_evaluator = BESSEvaluator(optimized_res)
envelope_res = bess_evaluator.evaluate_bess_performance()

import pandas as pd
from IPython.display import display, HTML

metrics_meta = [
    ("round_trip_energy_efficiency", "Round-Trip Energy Efficiency (RTE)", "eta_RTE", "{:.2%}"),
    ("coulombic_efficiency", "Coulombic Efficiency", "eta_C", "{:.2%}"),
    ("voltage_efficiency", "Voltage Efficiency", "eta_V", "{:.2%}"),
    ("usable_energy_capacity_wh", "Usable Energy Capacity", "E_usable", "{:.2f} Wh"),
    ("power_capability_w", "Power Capability", "P_max", "{:.2f} W"),
    ("thermal_response_delta_t", "Thermal Response Delta T", "Delta T", "{:.2f} K"),
    ("max_temperature_k", "Maximum Temperature", "T_max", "{:.2f} K"),
    ("depth_of_discharge", "Depth of Discharge", "DoD", "{:.2%}"),
    ("equivalent_full_cycles", "Equivalent Full Cycles", "EFC", "{:.4f}"),
    ("capacity_fade", "Capacity Fade", "F_Q", "{:.4e}"),
    ("cycle_life", "Estimated Cycle Life", "N_life", "{:.0f} cycles"),
    ("calendar_life_years", "Estimated Calendar Life", "t_life", "{:.1f} years"),
    ("levelized_cost_of_storage_usd_per_kwh", "Levelized Cost of Storage", "LCOS", "${:.4f}/kWh")
]

rows = []
for key, desc, sym, fmt in metrics_meta:
    val = envelope_res.get(key, 0.0)
    rows.append({"Metric": desc, "Symbol": sym, "Value": fmt.format(val)})

df_metrics = pd.DataFrame(rows)
display(HTML("<h3>NFPP BESS Robustness Evaluation Framework Metrics (paper.md aligned)</h3>"))
display(df_metrics)

## Stage 4: Wavelet-Domain Observability Datasets under Partial Observability
This section imports and displays the persisted Dataset 1 and Dataset 2 CSV files generated by `src/simulation/dataset.py`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

dataset_dir = Path("src/simulation")
dataset_1_path = dataset_dir / "dataset_1.csv"
dataset_2_path = dataset_dir / "dataset_2.csv"

if not dataset_1_path.exists() or not dataset_2_path.exists():
    print("Persisted datasets not found. Generating datasets via dataset.py...")
    from src.simulation.dataset import generate_experiments_dataset
    generate_experiments_dataset(n_scenarios=15, write_to_disk=True)

dataset_1 = pd.read_csv(dataset_1_path)
dataset_2 = pd.read_csv(dataset_2_path)

display(HTML("<h3>Dataset 1 (Scenario-Based Steady-State Network Realization)</h3>"))
display(dataset_1.head(15))

display(HTML("<h3>Dataset 2 (Event-Based Transient Realization with Normalized Waveforms)</h3>"))
display(dataset_2.head(15))

## Stage 5: Rigorous Statistical Validation Pipeline on Actual Wavelet Representations
This section executes the non-parametric statistical tests directly on the persisted Dataset 1 and Dataset 2.

In [ ]:
from src.statistics.dependence import run_dependence_analysis
print("Test 1 — Distance Correlation & HSIC Nonlinear Confirmation")
res_1 = run_dependence_analysis()


In [ ]:
from src.statistics.distribution import run_distribution_analysis
print("Test 2 — MMD Two-Sample Distribution Test")
res_2 = run_distribution_analysis()


In [ ]:
from src.statistics.permanova import run_permanova_analysis
print("Test 3 — PERMANOVA & PERMDISP Group Homogeneity Test")
res_3 = run_permanova_analysis()


In [ ]:
from src.statistics.equivalence import run_equivalence_analysis
print("Test 4 — TOST Practical Equivalence Test")
res_4 = run_equivalence_analysis()


In [ ]:
from src.statistics.observability import run_observability_analysis
print("Test 5 — Observability of Hidden State and Perturbations from Joint Wavelet/Spectral Representations")
res_5 = run_observability_analysis()
